## ingests data from external api

- source: https://earthquake.usgs.gov
- port: 443
- base path: /earthquakes/feed/v1.0
- url: summary/all_day.geojson


https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson

In [0]:
%python
import requests
import json
import datetime

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import catalog

w = WorkspaceClient()

conn = w.connections.get("earthquate_api_http_connection")
base_url = f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
%python
base_url

In [0]:
%python
dbutils.widgets.text("api_source", "summary/all_day.geojson")

api_source = dbutils.widgets.get("api_source")

print(api_source)

In [0]:
%python
url = f"{base_url}" + api_source

response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"Error in getting data from {url}")

data = response.json()

current_date = datetime.datetime.now().strftime("%Y-%m-%d-%HH-%MM-%SS")
print("current_date", current_date)
dbutils.fs.put(
    f"/Volumes/lakehouse/01_raw/raw/earthquake_data_{current_date}.json",
    json.dumps(data),
)